### HW4
#### Shuxiong Cai
For this assignment, I build a binary classification model to predict whether a driver will finish on the podium (top 3) in a race.

#### Q1:Build any model of your choice with tunable hyperparameters


In [0]:
import mlflow
import mlflow.sklearn
from pyspark.sql import functions as F
from sklearn.ensemble import RandomForestClassifier

df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True, inferSchema=True)
df_races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True, inferSchema=True)
df_drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True, inferSchema=True)
df_qualifying = spark.read.csv("/Volumes/gr5069/raw/f1_data/qualifying.csv", header=True, inferSchema=True)

In [0]:
df_model = (
    df_results
    .select("raceId", "driverId", "constructorId", "positionOrder", "grid")
    .join(
        df_races.select("raceId", "year", "round", "circuitId", "date"),
        on="raceId",
        how="left"
    )
    .join(
        df_drivers.select("driverId", "dob"),
        on="driverId",
        how="left"
    )
    .join(
        df_qualifying
        .select("raceId", "driverId", "position")
        .withColumnRenamed("position", "quali_position"),
        on=["raceId", "driverId"],
        how="left"
    )
    .withColumn("label", F.when(F.col("positionOrder") <= 3, 1).otherwise(0))
    .withColumn("age", F.year("date") - F.year("dob"))
    .select("label", "grid", "quali_position", "constructorId", "circuitId", "year", "round", "age")
    .dropna()
)

display(df_model)

In [0]:
pdf = df_model.toPandas()

X = pdf[["grid", "quali_position", "constructorId", "circuitId", "year", "round", "age"]]
y = pdf["label"]

print(X.head())
print(y.head())

In [0]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [0]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

rf.fit(X_train, y_train)

predictions = rf.predict(X_test)
print(predictions[:10])

#### Q2:Create an experiment setup where - for each run - you log:

the hyperparameters used in the model;

the model itself;

every possible metric from the model you chose;

at least two artifacts (plots, or csv files)

In [0]:
experiment_name = "/Users/sc5816@columbia.edu/HW4_f1_podium_prediction"
mlflow.set_experiment(experiment_name)

experiment = mlflow.get_experiment_by_name(experiment_name)
experimentID = experiment.experiment_id

print("Experiment ID:", experimentID)

In [0]:
def log_rf_classifier(experimentID, run_name, params, X_train, X_test, y_train, y_test):
    import os
    import tempfile
    import pandas as pd
    import matplotlib.pyplot as plt
    import mlflow
    import mlflow.sklearn

    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import (
        accuracy_score,
        precision_score,
        recall_score,
        f1_score,
        roc_auc_score,
        ConfusionMatrixDisplay
    )

    with mlflow.start_run(experiment_id=experimentID, run_name=run_name) as run:
        # Create model, train it, and create predictions
        rf = RandomForestClassifier(**params)
        rf.fit(X_train, y_train)
        predictions = rf.predict(X_test)
        probabilities = rf.predict_proba(X_test)[:, 1]

        # Log model
        mlflow.sklearn.log_model(rf, "random-forest-model")

        # Log params
        [mlflow.log_param(param, value) for param, value in params.items()]

        # Create metrics
        accuracy = accuracy_score(y_test, predictions)
        precision = precision_score(y_test, predictions, zero_division=0)
        recall = recall_score(y_test, predictions, zero_division=0)
        f1 = f1_score(y_test, predictions, zero_division=0)
        auc = roc_auc_score(y_test, probabilities)

        print("accuracy:", accuracy)
        print("precision:", precision)
        print("recall:", recall)
        print("f1:", f1)
        print("auc:", auc)

        # Log metrics
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1", f1)
        mlflow.log_metric("auc", auc)

        # Artifact 1: feature importance csv
        importance = pd.DataFrame(
            list(zip(X_train.columns, rf.feature_importances_)),
            columns=["Feature", "Importance"]
        ).sort_values("Importance", ascending=False)

        temp = tempfile.NamedTemporaryFile(prefix="feature-importance-", suffix=".csv", delete=False)
        temp_name = temp.name
        temp.close()
        try:
            importance.to_csv(temp_name, index=False)
            mlflow.log_artifact(temp_name, "feature-importance")
        finally:
            os.unlink(temp_name)

        # Artifact 2: confusion matrix plot
        fig, ax = plt.subplots(figsize=(5, 4))
        ConfusionMatrixDisplay.from_predictions(y_test, predictions, ax=ax)
        plt.title("Confusion Matrix")

        temp = tempfile.NamedTemporaryFile(prefix="confusion-matrix-", suffix=".png", delete=False)
        temp_name = temp.name
        temp.close()
        try:
            fig.savefig(temp_name, bbox_inches="tight")
            mlflow.log_artifact(temp_name, "plots")
        finally:
            plt.close(fig)
            os.unlink(temp_name)

        return run.info.run_id

In [0]:
params = {
    "n_estimators": 100,
    "max_depth": 5,
    "random_state": 42
}

run_id = log_rf_classifier(
    experimentID,
    "Baseline RF Classifier",
    params,
    X_train,
    X_test,
    y_train,
    y_test
)

print("Run ID:", run_id)

I created an MLflow experiment with my RandomForestClassifier. For each run,  I log the hyperparameters, the trained model, multiple classification metrics, and at least two artifacts.


The two artifacts I log are:
- a feature importance CSV file
- a confusion matrix plot

#### Q3:Track your MLFlow experiment and run at least 10 experiments with different parameters each

In [0]:
param_list = [
    {"n_estimators": 50, "max_depth": 3, "random_state": 42},
    {"n_estimators": 50, "max_depth": 5, "random_state": 42},
    {"n_estimators": 50, "max_depth": 10, "random_state": 42},
    {"n_estimators": 100, "max_depth": 3, "random_state": 42},
    {"n_estimators": 100, "max_depth": 5, "random_state": 42},
    {"n_estimators": 100, "max_depth": 10, "random_state": 42},
    {"n_estimators": 200, "max_depth": 3, "random_state": 42},
    {"n_estimators": 200, "max_depth": 5, "random_state": 42},
    {"n_estimators": 200, "max_depth": 10, "random_state": 42},
    {"n_estimators": 300, "max_depth": 5, "random_state": 42}
]

run_ids = []

for i, params in enumerate(param_list, start=1):
    run_id = log_rf_classifier(
        experimentID,
        f"RF_Run_{i}",
        params,
        X_train,
        X_test,
        y_train,
        y_test
    )
    run_ids.append(run_id)

print(run_ids)

#### Q4:Select your best model run and explain why


I selected `RF_Run_6` as my best model run. In determining the best model, AUC was mostly considered since we are dealing with a binary classification dataset, which means that AUC reflects performance overall.
`RF_Run_6` have the one of the highest AUC scores in the experiment set (`0.918`). On the other hand, it performed slightly better compared to the other tied model run on all the other measures, including accuracy (0.900), F1 score (0.615), precision (0.672), and recall (0.568).

Therefore, based on its overall performance with respect to metrics, I selected RF_Run_6 as the best performing model.

#### Q5
The screenshots required for Question 5 are included in the `screenshots` folder in my GitHub classroom repository. These include:
- the MLflow homepage,
- the detailed page for my selected best run (`RF_Run_6`),
- and the artifacts page.